# Cuaderno asociado al TFG

Este cuaderno forma parte del repositorio asociado al Trabajo de Fin de Grado **“Impacto de las Telecomunicaciones en la Agricultura 4.0”**.

Por motivos de confidencialidad, los archivos de datos reales no se incluyen en el repositorio. El código se mantiene como referencia metodológica y está preparado para trabajar con archivos Excel equivalentes ubicados en la carpeta `Datos/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

In [ ]:
# Cargar datos
df = pd.read_excel("../Datos/Datos_cultivos.xlsx")

# Copia de trabajo
df = df.copy()

print("Dimensiones:", df.shape)
df.head()

In [ ]:
#inspeccion de datos
df.info()

En humedad tenemos cinco nulos correspondientes al cultivo del tomate ya que este no tiene humedad al cosecharse

In [ ]:
#Comprobación de cultivos
df["Cultivo"].value_counts()


In [ ]:
#LIMPIEZA
#Eliminar espacios en nombres
df.columns = df.columns.str.strip()


In [ ]:
#aseguramos las Flags como binarias

TECH_FLAGS = [
    "GPS",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

for col in TECH_FLAGS:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)


In [ ]:
#Variables:
# Consumo real por hectárea
df["Consumo_real_L_ha"] = df["Consumo_L_ha"] * df["Pasadas_ha"]

# Gasoil por tonelada (KPI energético clave)
df["Gasoil_L_por_t"] = df["Consumo_real_L_ha"] / df["Rend_t_ha"]

# Evitar infinitos
df["Gasoil_L_por_t"] = df["Gasoil_L_por_t"].replace([np.inf, -np.inf], np.nan)


In [ ]:
df[["Consumo_real_L_ha","Gasoil_L_por_t"]].describe()


Consumo real litros por hectárea: 
media de 155 l/ha, max 415 L/ha

Gasoil litros por tonelada: 
Media de 14,5L/t, mediana 8,36 L/t.
Cmo la media es bastante mayor que la mediana, la distribución está sesgada a la derecha. 


In [ ]:
#Análisis descriptivo por cultivo
for cultivo in df["Cultivo"].unique():
    print("\n==============================")
    print("Cultivo:", cultivo)
    sub = df[df["Cultivo"] == cultivo]
    
    print("n =", len(sub))
    display(sub[["Rend_t_ha",
                 "Consumo_real_L_ha",
                 "Gasoil_L_por_t"]].describe())


LECTURA DE RESULTADOS

MAIZ
Rendimiento medio: 15,55 t/ha
Consumo real: 90 L/ha
KPI energético: 5,88 L/t
Desviación baja: sistema bastante homogéneo
Maíz es energéticamente eficiente y estable.


TOMATE
Rendimiento: 108 t/ha 
Consumo: 294 L/ha
KPI: 2.85 L/t
Aunque tenemos pocos datos, n=5


ARROZ
Rendimiento: 8,07 t/ha
Consumo: 174,27 L/ha
KPI: 21,78 L/t
Desviación 3.24: variabilidad interesante
Arroz es muchísimo más intensivo energéticamente.
Aquí es donde puede verse impacto tecnológico.


In [ ]:
#Tech index: 
df["tech_index"] = df[TECH_FLAGS].sum(axis=1)
df["tech_index"].describe()


Medai de 4,56 sobre 6, lo que implica que la mayoría de explotaciones ya están bastante tecnificadas

In [ ]:
df.groupby(["Cultivo"])["tech_index"].describe()


In [ ]:
#Correlacion
for cultivo in ["Maiz", "Arroz"]:
    print("\n==============================")
    print("Cultivo:", cultivo)
    sub = df[df["Cultivo"] == cultivo]
    
    corr = sub[["tech_index", "Gasoil_L_por_t"]].corr()
    print(corr)


Gran correlación, grupo tecnológicos bien definidos

Más tecnología: menos solapes
RTK reduce pasadas
Corte por tramos reduce redundancias y reduce el uso de insumios
Abono variable reduce el gasto
Todo esto unido a una mejor planificación da lugar a menos horas necesarias de trabajo.

In [ ]:
#Visualización
import matplotlib.pyplot as plt

for cultivo in ["Maiz", "Arroz"]:
    sub = df[df["Cultivo"] == cultivo]
    
    plt.figure()
    plt.scatter(sub["tech_index"], sub["Gasoil_L_por_t"])
    plt.xlabel("tech_index")
    plt.ylabel("Gasoil_L_por_t")
    plt.title(f"{cultivo}")
    plt.show()


Vemos que el tech_index solo toma valores de 1,4,5,6


In [ ]:
for cultivo in ["Maiz", "Arroz"]:
    print("\n==============================")
    print("Cultivo:", cultivo)
    sub = df[df["Cultivo"] == cultivo]
    
    print("Corr tech vs Consumo_real_L_ha")
    print(sub[["tech_index", "Consumo_real_L_ha"]].corr())
    
    print("\nCorr tech vs Rend_t_ha")
    print(sub[["tech_index", "Rend_t_ha"]].corr())


MAIZ
tecnología vs consumo/ha: correlación = -0,85. 
Muy fuerte, a mayor tecnología, menor consumo por hectárea.
Tecnología vs Rendimiento: correlación: +0,91
A mayor tecnología, mayor rendimiento
La tecnología reduce el consumo así como aumenta el rendimiento

Arroz: 
tecnología vs consumo/ha: correlación = -0,89. 
Muy fuerte, a mayor tecnología, menor consumo por hectárea.
Tecnología vs Rendimiento: correlación: +0,92
A mayor tecnología, mayor rendimiento. Mayor impacto aún que en el maiz. 


In [ ]:
#Vamos a ver que tecnología tiene mayor correlación individual con eficiencia: 
for cultivo in ["Maiz", "Arroz"]:
    print("\n==============================")
    print("Cultivo:", cultivo)
    sub = df[df["Cultivo"] == cultivo]
    
    print(sub[TECH_FLAGS + ["Gasoil_L_por_t"]].corr()["Gasoil_L_por_t"])


Interpretación de las correlaciones por tecnología: 

-GPS=NaN.
Ya que todas las explotaciones tienen GPS, tecnología la cuál ya se lleva utilizando desde hace bastantes años, igual no tan sofisticada como en la actualidad pero de uso ya implantado. Una variable constante no tiene correlación, varianza cero. 

MAIZ: 
las tecnologías que más reducen consumo energético por tonelada son: RTK, ISOBUS, CORTE POR TRAMOS

ARROZ: 
RTK e ISOBUs dominan el comportamiento enegético. 

In [ ]:
#Modelo OLS para cultivo

# Seleccionamos Maíz
df_maiz = df[df["Cultivo"] == "Maiz"].copy()

# Variables explicativas
X = df_maiz[TECH_FLAGS]

# Variable dependiente
y = df_maiz["Gasoil_L_por_t"]

# Añadimos constante
X = sm.add_constant(X)

# Modelo OLS
model_maiz = sm.OLS(y, X).fit()

print(model_maiz.summary())


Métricas generales: 
R^2 = 0.884
Adj R^2 = 0.863
F-stat muy significativo
El modelo explica bastante bien la variabilidad.

Condition number: 5,55e+17
Por lo que tiene "strong multicollinearity problems", lo que quiere decir que las tecnologías están altamente correlacionadas entre sí, se adoptan juntas. 
Por eso, RTK e ISOBUS tienen exactamente el mismo coeficiente. 

Las más significativas en el maiz son: 
p<0,05 RTK e ISOBUS

In [ ]:
#Modelo para arroz
df_arroz = df[df["Cultivo"] == "Arroz"].copy()

X = df_arroz[TECH_FLAGS]
y = df_arroz["Gasoil_L_por_t"]

X = sm.add_constant(X)

model_arroz = sm.OLS(y, X).fit()

print(model_arroz.summary())


ARROZ:

R^2 = 0.921
Adj R^2= 0.909
De nuevo tenemos una alta multicolinealidad.
TOdas las tecnologías salen significativas, más aúnn el corte por tramos, están altamente relacionadas.

A mayor tecnología, menor consumo


In [ ]:
#Modelo simple con tech_index

df_maiz = df[df["Cultivo"] == "Maiz"]

X = df_maiz[["tech_index"]]
y = df_maiz["Gasoil_L_por_t"]

X = sm.add_constant(X)

model_simple_maiz = sm.OLS(y, X).fit()

print(model_simple_maiz.summary())


Condition number=12,8, sin colinealidad

Por cada tecnología adicional adoptada: el consumo energético disminuye 0,54L por Tonelada (coeficiente tech_index)

Por ejemplo si pasamos de tech_index de 1 a 6: reducción esperada: 5*0,54 = 2,5L/t

El nivel tecnológico agregado explica el 88% de la variabilidad en la intensidad energética de producción. Cada incremento unitario en el índice tecnológico reduce el consumo energético en aprox 0,54L por tonelada

In [ ]:
#Igual para el arroz
df_arroz = df[df["Cultivo"] == "Arroz"]

X = df_arroz[["tech_index"]]
y = df_arroz["Gasoil_L_por_t"]

X = sm.add_constant(X)

model_simple_arroz=sm.OLS(y, X).fit()

print(model_simple_arroz.summary())


Ahora aún más significativo, el consumo disminuye 1,62 L por tonelada (coeficiente tech_index = -1,6192)

La tecnología tiene un impacto 3 veces mayor en el arroz.

El efecto es aún más pronunciado, con mayor reducción estimada por tonelada. Por cada tecnología adicional adoptada, explicando el 97% de la variabilidad

In [ ]:
#RIDGE (regularización)

df_maiz = df[df["Cultivo"] == "Maiz"]

X = df_maiz[TECH_FLAGS]
y = df_maiz["Gasoil_L_por_t"]

# Pipeline con estandarización
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5))
])

ridge_model.fit(X, y)

print("Alpha óptimo:", ridge_model.named_steps["ridge"].alpha_)

coefs = ridge_model.named_steps["ridge"].coef_
for name, coef in zip(TECH_FLAGS, coefs):
    print(name, round(coef, 3))

# R2 con validación cruzada
scores = cross_val_score(ridge_model, X, y, cv=5, scoring="r2")
print("R2 medio CV:", scores.mean())


In [ ]:
#Random forest

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X, y)

# R2 con CV
scores_rf = cross_val_score(rf, X, y, cv=5, scoring="r2")
print("R2 medio RF:", scores_rf.mean())

# Importancia por permutación
perm = permutation_importance(rf, X, y, n_repeats=20, random_state=42)

for i in perm.importances_mean.argsort()[::-1]:
    print(TECH_FLAGS[i], round(perm.importances_mean[i], 3))


El modelo lineal explica bien la estructura completa, pero su capacidad predictiva fuera de muestra es limitada debido al tamaño muestral reducido y a la estructura discreta del índica tecnológico.

La regularización no mejora la capacidad predictiva debido a la estructura altamente correlacionada y tamaño muestral reducido. 

Modelos no lineales como RandomForest no aportan mejorar significativas, lo que sugiere que la relación entre nivel tecnológico y eficiencia energética es altamente lineal.


In [ ]:
#Arroz

# Filtrar arroz
df_arroz = df[df["Cultivo"] == "Arroz"]

X = df_arroz[TECH_FLAGS]
y = df_arroz["Gasoil_L_por_t"]

ridge_model_arroz = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5))
])

ridge_model_arroz.fit(X, y)

print("Alpha óptimo:", ridge_model_arroz.named_steps["ridge"].alpha_)

coefs = ridge_model_arroz.named_steps["ridge"].coef_
for name, coef in zip(TECH_FLAGS, coefs):
    print(name, round(coef, 3))

scores = cross_val_score(ridge_model_arroz, X, y, cv=5, scoring="r2")
print("R2 medio CV:", scores.mean())


In [ ]:
#Random Forest

rf_arroz = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_arroz.fit(X, y)

scores_rf = cross_val_score(rf_arroz, X, y, cv=5, scoring="r2")
print("R2 medio RF:", scores_rf.mean())

perm = permutation_importance(
    rf_arroz, X, y,
    n_repeats=20,
    random_state=42
)

for i in perm.importances_mean.argsort()[::-1]:
    print(TECH_FLAGS[i], round(perm.importances_mean[i], 3))


En este caso, si generaliza algo mejor fuera de la muestra, pero sigue siendo principalmente lineal la relación con la tecnología
Dominan tanto RTK como ISOBUS

Vamos a ver ahora que tecnología está asociada a un mayor rendimiento de producción


In [ ]:
df_maiz = df[df["Cultivo"] == "Maiz"]

X = df_maiz[["tech_index"]]
y = df_maiz["Rend_t_ha"]

X = sm.add_constant(X)

model_rend_maiz = sm.OLS(y, X).fit()
print(model_rend_maiz.summary())


Coeficiente tech_index: +0,629t/ha
El rendimiento aumento 0,63 toneladas por hectárea

Parte de la mejora energética viene por mayor rendimiento

In [ ]:
df_arroz = df[df["Cultivo"] == "Arroz"]

X = df_arroz[["tech_index"]]
y = df_arroz["Rend_t_ha"]

X = sm.add_constant(X)

model_rend_arroz = sm.OLS(y, X).fit()
print(model_rend_arroz.summary())


El rendimiento aumenta 0,25t/ha
El impacto energético es mayor que el productivo, gran parte de la mejora viene por reducción directa de consumo por ha

In [ ]:
#ridge


df_maiz = df[df["Cultivo"] == "Maiz"]

X = df_maiz[TECH_FLAGS]
y = df_maiz["Rend_t_ha"]

ridge_rend_maiz = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=np.logspace(-3,3,100), cv=5))
])

ridge_rend_maiz.fit(X, y)

print("Alpha óptimo:", ridge_rend_maiz.named_steps["ridge"].alpha_)

coefs = ridge_rend_maiz.named_steps["ridge"].coef_
for name, coef in zip(TECH_FLAGS, coefs):
    print(name, round(coef,3))

print("R2 CV:", cross_val_score(ridge_rend_maiz, X, y, cv=5, scoring="r2").mean())


In [ ]:
#Random forest
rf_rend_maiz = RandomForestRegressor(n_estimators=200, random_state=42)
rf_rend_maiz.fit(X, y)

print("R2 CV:", cross_val_score(rf_rend_maiz, X, y, cv=5, scoring="r2").mean())

perm = permutation_importance(rf_rend_maiz, X, y, n_repeats=20, random_state=42)

for i in perm.importances_mean.argsort()[::-1]:
    print(TECH_FLAGS[i], round(perm.importances_mean[i],3))


La tecnología que más se asocia con un aumento de rendimmiento es Corte por tramos
Las tecnologías de aplicación variable también tienen impacto relevante
RTL e ISOBUS influyen menos en producción que en eficiencia energética

En Random Forest, coincide con Ridge, Corte por tramos es la técnología más influyente
El aumento de rendimiento está más asociado a tecnologías de gestión agronómica que a tecnologías de posicionamiento. 

In [ ]:
#ridge


df_arroz = df[df["Cultivo"] == "Arroz"]

X = df_arroz[TECH_FLAGS]
y = df_arroz["Rend_t_ha"]

ridge_rend_arroz = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=np.logspace(-3,3,100), cv=5))
])

ridge_rend_arroz.fit(X, y)

print("Alpha óptimo:", ridge_rend_arroz.named_steps["ridge"].alpha_)

coefs = ridge_rend_arroz.named_steps["ridge"].coef_
for name, coef in zip(TECH_FLAGS, coefs):
    print(name, round(coef,3))

print("R2 CV:", cross_val_score(ridge_rend_arroz, X, y, cv=5, scoring="r2").mean())


In [ ]:

rf_rend_arroz = RandomForestRegressor(n_estimators=200, random_state=42)
rf_rend_arroz.fit(X, y)

print("R2 CV:", cross_val_score(rf_rend_arroz, X, y, cv=5, scoring="r2").mean())

perm = permutation_importance(rf_rend_arroz, X, y, n_repeats=20, random_state=42)

for i in perm.importances_mean.argsort()[::-1]:
    print(TECH_FLAGS[i], round(perm.importances_mean[i],3))


En arroz, tambien tienen un gran impacto las tecnologías agronómicas, sobretodo en Ridge
Pero en este caso el efecto productivo es menor que el energético.

In [ ]:
#Impacto tecnológico sobre la productividad por hectárea. 

for cultivo in ["Maiz", "Arroz"]:
    print("\n" + "="*60)
    print("Cultivo:", cultivo)

    sub = df[df["Cultivo"] == cultivo].copy()

    X = sm.add_constant(sub[["tech_index"]])
    y = sub["Productividad_ha_h"]

    model_prod = sm.OLS(y, X).fit()
    print(model_prod.summary())


In [ ]:
#Ridge

for cultivo in ["Maiz", "Arroz"]:
    print("\n" + "="*60)
    print("RIDGE - Cultivo:", cultivo)

    sub = df[df["Cultivo"] == cultivo].copy()
    X = sub[TECH_FLAGS]
    y = sub["Productividad_ha_h"]

    ridge_prod = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=np.logspace(-3,3,100), cv=5))
    ])

    ridge_prod.fit(X, y)

    print("Alpha óptimo:", ridge_prod.named_steps["ridge"].alpha_)

    coefs = ridge_prod.named_steps["ridge"].coef_
    for name, coef in zip(TECH_FLAGS, coefs):
        print(name, round(coef,3))

    print("R2 CV:", cross_val_score(ridge_prod, X, y, cv=5, scoring="r2").mean())


In [ ]:
#Random forest
for cultivo in ["Maiz", "Arroz"]:
    print("\n" + "="*60)
    print("RANDOM FOREST - Cultivo:", cultivo)

    sub = df[df["Cultivo"] == cultivo].copy()
    X = sub[TECH_FLAGS]
    y = sub["Productividad_ha_h"]

    rf_prod = RandomForestRegressor(n_estimators=300, random_state=42)
    rf_prod.fit(X, y)

    print("R2 CV:", cross_val_score(rf_prod, X, y, cv=5, scoring="r2").mean())

    perm = permutation_importance(rf_prod, X, y, n_repeats=30, random_state=42)

    for i in perm.importances_mean.argsort()[::-1]:
        print(TECH_FLAGS[i], round(perm.importances_mean[i],3))


La productividad por hectárea está casi perfectamente determinada por tech_index 

por ejemplo en maiz, cada tecnología adicional aumenta la productividad en 0,06 ha/h aprox

En el arroz es casi igual el impacto que tiene cada tecnología adicional.

En ambos tanto el rtk como el isobus son las más relacionadas con la eficiencia operativa

In [ ]:
df.groupby("tech_index")["Productividad_ha_h"].mean()


A mayor nivel tecnológico, la productividad operativa aumenta de forma clara

TOMATE

In [ ]:
#Análisis descriptivo
sub_tom = df[df["Cultivo"] == "Tomates"].copy()

print("n =", len(sub_tom))
print("\nTech index:")
print(sub_tom["tech_index"].value_counts())

print("\nRendimiento:")
print(sub_tom["Rend_t_ha"].describe())

print("\nConsumo por tonelada:")
print(sub_tom["Gasoil_L_por_t"].describe())


In [ ]:
#Relación tech_index rendimiento

X = sm.add_constant(sub_tom[["tech_index"]])
y = sub_tom["Rend_t_ha"]

model_tom = sm.OLS(y, X).fit()
print(model_tom.summary())


In [ ]:

plt.figure()
plt.scatter(sub_tom["tech_index"], sub_tom["Rend_t_ha"])
plt.xlabel("tech_index")
plt.ylabel("Rend_t_ha")
plt.title("Tomate")
plt.show()


Al ser tan pocas observacines no podemos decir claramente que ninguna tecnología afecte directamente en la producción más que otra

Si bien la producción con respecto al año que no hemos utilizado técnología ha aumentado, no podemos sacar conclusiones muy claras

Con lo que también sacar en conclusión que en este cultivo depende más aún de la climatologia y la diferencia entre los años


In [ ]:
#consumo por tonelada
X = sm.add_constant(sub_tom[["tech_index"]])
y = sub_tom["Gasoil_L_por_t"]
print(y)

In [ ]:
for tech in ["RTK", "ISOBUS", "Corte_tramos", "Siembra_variable", "Abono_variable"]:
    print("\n", tech)
    print(sub_tom.groupby(tech)["Rend_t_ha"].mean())


In [ ]:
import statsmodels.api as sm

sub_tom = df[df["Cultivo"] == "Tomates"].copy()

for tech in ["RTK", "ISOBUS", "Corte_tramos", "Siembra_variable", "Abono_variable"]:
    
    print("\n==========================")
    print("Tecnología:", tech)
    
    X = sm.add_constant(sub_tom[[tech]])
    y = sub_tom["Rend_t_ha"]
    
    model = sm.OLS(y, X).fit()
    print(model.summary())


Todas las técnologias, RTK, ISOBUS, corte tramos y abono variable tienen los mismos coeficientes, ya que todas las tecnologías aparecen en las mismas parcelas



In [ ]:
#Variabilidad entre cultivos
df.groupby("Cultivo")["Rend_t_ha"].std()


El tomate tiene una variabilidad de rendimiento mucho mayor que los otros cultivos
Esto sugiere que factores externos, no al alcance del agricultor o del desarrollo tecnológico, son bastante influyentes en la producción anual de tomates



In [ ]:
#coeficiente de variación

cv = df.groupby("Cultivo")["Rend_t_ha"].agg(["mean", "std"])
cv["coef_variacion"] = cv["std"] / cv["mean"]

cv


El coeficiente de variación es más del doble al observado en los otros dos cultivos, lo que indica una mayor dispersión relativa y sugiere una mayor sensibilidad a factores externos no modelizados.

GRÁFICAS

In [ ]:
#tech index vs rendimiento por cultivo


for cultivo in ["Maiz", "Tomates","Arroz"]:
    sub = df[df["Cultivo"] == cultivo]
    
    plt.figure()
    plt.scatter(sub["tech_index"], sub["Rend_t_ha"])
    plt.xlabel("tech_index")
    plt.ylabel("Rendimiento (t/ha)")
    plt.title(f"{cultivo} - Tecnología vs Rendimiento")
    #plt.savefig("Tecnologia_vs_rendimiento.png", dpi=300, bbox_inches="tight")

    plt.show()


In [ ]:
#tech index vs litros de gasoil por tonelada

for cultivo in ["Maiz", "Tomates", "Arroz"]:
    sub = df[df["Cultivo"] == cultivo]
    
    plt.figure()
    plt.scatter(sub["tech_index"], sub["Gasoil_L_por_t"])
    plt.xlabel("tech_index")
    plt.ylabel("Gasoil real (L/t)")
    plt.title(f"{cultivo} - Tecnología vs Consumo por tonelada")
    #plt.savefig("Tecnologia_vs_consumo.png", dpi=300, bbox_inches="tight")

    plt.show()


In [ ]:
#tech index vs productividad hectárea hora
for cultivo in ["Maiz", "Arroz", "Tomates"]:
    sub = df[df["Cultivo"] == cultivo]
    
    plt.figure()
    plt.scatter(sub["tech_index"], sub["Productividad_ha_h"])
    plt.xlabel("tech_index")
    plt.ylabel("Productividad (ha/h)")
    plt.title(f"{cultivo} - Tecnología vs Productividad")
    plt.show()


In [ ]:
#comparación variabilidad
df.boxplot(column="Rend_t_ha", by="Cultivo")
plt.title("Distribución del rendimiento por cultivo")
plt.suptitle("")
plt.ylabel("Rendimiento (t/ha)")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12,4))

cultivos = ["Arroz", "Maiz", "Tomates"]

for ax, cultivo in zip(axes, cultivos):
    sub = df[df["Cultivo"] == cultivo]
    ax.boxplot(sub["Rend_t_ha"])
    ax.set_title(cultivo)
    ax.set_ylabel("Rendimiento (t/ha)")

plt.suptitle("Distribución del rendimiento por cultivo")
plt.tight_layout()

plt.savefig("Rendimiento_por_cultivo.png", dpi=300, bbox_inches="tight")

plt.show()


In [ ]:
#promedio por nivel tecnológico
mean_by_tech = df.groupby(["Cultivo", "tech_index"])["Rend_t_ha"].mean().unstack(0)

mean_by_tech.plot(marker="o")
plt.xlabel("tech_index")
plt.ylabel("Rendimiento medio (t/ha)")
plt.title("Rendimiento medio por nivel tecnológico")
plt.show()


In [ ]:
#tecnologia vs consumo específico (L/t) 

plt.figure(figsize=(8,6))

cultivos = ["Arroz", "Maiz", "Tomates"]
colores = {"Arroz":"tab:blue", "Maiz":"tab:orange", "Tomates":"tab:green"}

for cultivo in cultivos:
    sub = df[df["Cultivo"] == cultivo]
    
    # Scatter
    plt.scatter(sub["tech_index"], 
                sub["Gasoil_L_por_t"], 
                alpha=0.7, 
                label=cultivo,
                color=colores[cultivo])
    
    # Línea de tendencia
    z = np.polyfit(sub["tech_index"], sub["Gasoil_L_por_t"], 1)
    p = np.poly1d(z)
    x_vals = np.linspace(sub["tech_index"].min(), sub["tech_index"].max(), 100)
    plt.plot(x_vals, p(x_vals), color=colores[cultivo])

plt.xlabel("tech_index")
plt.ylabel("Consumo específico (L/t)")
plt.title("Tecnología vs Consumo energético específico")
plt.legend()
plt.tight_layout()

plt.savefig("Tecnologia_vs_consumo_cultivos.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
#impacto relativo (% cambio) del rendimiento por punto tecnológico


plt.figure(figsize=(8,6))

for cultivo in ["Arroz", "Maiz", "Tomates"]:
    sub = df[df["Cultivo"] == cultivo].copy()
    
    # Modelo simple rendimiento ~ tech_index
    X = sm.add_constant(sub["tech_index"])
    y = sub["Rend_t_ha"]
    model = sm.OLS(y, X).fit()
    
    coef = model.params["tech_index"]
    media = sub["Rend_t_ha"].mean()
    
    impacto_pct = (coef / media) * 100
    
    plt.bar(cultivo, impacto_pct)

plt.ylabel("Incremento % del rendimiento por punto tech_index")
plt.title("Impacto relativo de la tecnología sobre el rendimiento")
plt.tight_layout()

plt.savefig("Impacto_relativo_tecnologia.png", dpi=300, bbox_inches="tight")

plt.show()

Conclusión del análisis técnico

Los resultados obtenidos muestran que, el los cultivos de maíz y arroz, el incremento en el nivel tecnológico, tech_index, se asocia de forma consistente con una mejora en el rendimiento por hectárea, una reducción del consumo de energético por tonelada producida y un aumento de la productividad operativa (ha/h). 
Los modelos OLS, Ridge y Random Forest presentan ajustes elevados y coherentes entre sí, destacando especialmente el papel de tecnologías de posicionamiento de alta precisión (RTK) y sistemas de comunicación tractor-apero (ISOBUS) como las variables con mayor contribución relativa. 

En el tomate, sin embargo, no se observa una relación estadísticamente robusta entre las tecnologías analizadas y el rendimiento productivo. EL reducido tamaño muestran (n=5), la aplicación conjunta de varias tecnologíasy la elevada variabilidad interanual del rendimiento limitan la capacidad de inferencia. El coeficiente de variación del tomate es significativamente superior al de maíz y arroz, lo que sugiere una mayor dispersión relativa y posiblemente una mayor influencia de factores externos no modelizados. 

En conjunto, el análisis técnico evidencia que el impacto de la digitalización agrícola no es homogéneo entre cultivos. Es más claro y  estructural en sistemas productivos con menor variabilidad interna como maíz y arroz, mientras que en el tomate la señal tecnológica queda parcialmente diluida por la variabilidad propia del cultivo